In [1]:
# ============================================================
# PHASE 23E — DATA LOADERS FOR FINE-TUNING
# ============================================================
#
# REASONING:
# Fine-tuning requires training images and validation images.
#
# We recreate the DataLoaders here so this notebook is
# independent of variables created in previous notebooks.
#
# Training:
#   Uses augmentation because we want the model to generalize.
#
# Validation:
#   Uses NO random augmentation because validation must remain
#   consistent and unbiased.
#
# Both use the same 224x224 size and ImageNet normalization
# used during our original Swin-Tiny training.
# ============================================================

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Training transformations
#
# Augmentation creates slightly different versions of training
# images so the model does not simply memorize the training set.
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation transformations
#
# NO random augmentation is used here.
# ------------------------------------------------------------

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    "../dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "../dataset/val",
    transform=val_transform
)


# ------------------------------------------------------------
# Create DataLoaders
#
# batch_size=32:
#   Same batch size used during our original experiments.
#
# shuffle=True for training:
#   Prevents the model from seeing training samples in the
#   exact same order every epoch.
#
# shuffle=False for validation:
#   Validation order does not need randomization.
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


# ------------------------------------------------------------
# Verify everything
# ------------------------------------------------------------

print("\nTraining images :", len(train_dataset))
print("Validation images:", len(val_dataset))

print("\nClasses:")
print(train_dataset.classes)

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nTraining batches :", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n✅ DataLoaders ready for fine-tuning.")

Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU

Training images : 8017
Validation images: 1998

Classes:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Class mapping:
{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}

Training batches : 251
Validation batches: 63

✅ DataLoaders ready for fine-tuning.


In [2]:
# ============================================================
# PHASE 23E — PRUNED MODEL FINE-TUNING
# ============================================================
#
# GOAL:
# Fine-tune the 10% and 20% pruned Swin-Tiny models and check
# whether their classification performance can recover.
#
# REASONING:
# After pruning, some weights are forced to zero.
# Fine-tuning allows the remaining weights to adapt.
#
# IMPORTANT:
# We create a MASK for the pruned weights.
# After every optimizer update, the mask is reapplied so that
# pruned weights remain exactly zero.
#
# Otherwise, the optimizer could update those zero weights again,
# effectively undoing the pruning.
#
# We use:
#   - AdamW
#   - small learning rate = 1e-5
#   - 5 fine-tuning epochs
#   - class-weighted CrossEntropyLoss
#   - Macro F1 for model selection
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
import copy
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# STEP 1 — CREATE THE SWIN-TINY ARCHITECTURE
# ============================================================
#
# REASONING:
# The .pth file contains learned parameters, but PyTorch first
# needs the correct model architecture into which those parameters
# will be loaded.
# ============================================================

model = models.swin_t(weights=None)

num_classes = 7

in_features = model.head.in_features

model.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

print("✅ Swin-Tiny architecture created.")


# ============================================================
# STEP 2 — LOAD THE ORIGINAL TRAINED CHECKPOINT
# ============================================================
#
# REASONING:
# We start from our already-trained Swin-Tiny rather than
# training from scratch.
# ============================================================

checkpoint_path = "../models/swin_tiny_best.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        state_dict = checkpoint

else:
    state_dict = checkpoint

model.load_state_dict(state_dict)

model.eval()

print("✅ Original Swin-Tiny checkpoint loaded.")


# ============================================================
# STEP 3 — CALCULATE CLASS WEIGHTS
# ============================================================
#
# REASONING:
# HAM10000 is highly imbalanced.
#
# Class weighting gives greater loss importance to minority
# classes and prevents the majority class from dominating training.
#
# Formula:
#
# class weight = total_samples / (number_of_classes × class_count)
#
# ============================================================

class_counts = torch.bincount(
    torch.tensor(train_dataset.targets),
    minlength=num_classes
).float()

class_weights = len(train_dataset) / (
    num_classes * class_counts
)

class_weights = class_weights.to(device)

print("\nClass counts:")
print(class_counts.cpu().numpy())

print("\nClass weights:")
for class_name, weight in zip(
    train_dataset.classes,
    class_weights.cpu().numpy()
):
    print(f"{class_name:6s}: {weight:.4f}")


# ============================================================
# STEP 4 — IDENTIFY PRUNABLE PARAMETERS
# ============================================================
#
# REASONING:
# We use unstructured pruning.
#
# Individual values inside 2D Linear weight matrices are removed.
#
# We exclude:
#   - biases
#   - normalization parameters
#
# because these parameters have different roles and should not
# be blindly pruned in this first experiment.
# ============================================================

def get_prunable_parameters(model):

    prunable = []

    for name, param in model.named_parameters():

        if (
            "weight" in name
            and param.requires_grad
            and param.ndim == 2
            and "norm" not in name.lower()
        ):
            prunable.append((name, param))

    return prunable


print(
    "\nPrunable weight tensors:",
    len(get_prunable_parameters(model))
)


# ============================================================
# STEP 5 — ESTIMATE GLOBAL PRUNING THRESHOLD
# ============================================================
#
# REASONING:
# Magnitude-based pruning uses |weight|.
#
# Smaller magnitude → candidate for removal.
#
# We estimate the threshold using samples because Swin-Tiny
# contains millions of parameters.
# ============================================================

def estimate_threshold(model, pruning_percent):

    samples = []

    MAX_SAMPLES_PER_LAYER = 100_000

    for name, param in get_prunable_parameters(model):

        values = (
            param.detach()
            .abs()
            .flatten()
            .cpu()
        )

        if values.numel() > MAX_SAMPLES_PER_LAYER:

            indices = torch.randperm(
                values.numel()
            )[:MAX_SAMPLES_PER_LAYER]

            values = values[indices]

        samples.append(values)

    sampled_weights = torch.cat(samples)

    threshold = torch.quantile(
        sampled_weights,
        pruning_percent / 100
    )

    return threshold.item()


# ============================================================
# STEP 6 — CREATE A PRUNED MODEL + MASK
# ============================================================
#
# REASONING:
# The MASK records which weights are allowed to remain.
#
# 1 → keep weight
# 0 → permanently prune weight
#
# The original model is copied so our original Swin-Tiny remains
# untouched.
# ============================================================

def create_pruned_model(original_model, pruning_percent):

    pruned_model = copy.deepcopy(
        original_model
    )

    threshold = estimate_threshold(
        pruned_model,
        pruning_percent
    )

    masks = {}

    with torch.no_grad():

        for name, param in get_prunable_parameters(
            pruned_model
        ):

            mask = (
                param.abs() > threshold
            ).float()

            param.mul_(mask)

            masks[name] = mask

    return pruned_model, masks, threshold


# ============================================================
# STEP 7 — APPLY MASK
# ============================================================
#
# REASONING:
# Optimizers can change zero-valued parameters.
#
# Therefore, after every optimizer update, we multiply the
# weights by their pruning masks again.
#
# This guarantees that pruned weights remain zero.
# ============================================================

def apply_masks(model, masks):

    with torch.no_grad():

        for name, param in model.named_parameters():

            if name in masks:

                param.mul_(
                    masks[name].to(param.device)
                )


# ============================================================
# STEP 8 — CALCULATE ACTUAL SPARSITY
# ============================================================

def calculate_sparsity(model):

    total = 0
    zeros = 0

    for name, param in get_prunable_parameters(model):

        total += param.numel()

        zeros += (
            param == 0
        ).sum().item()

    return (zeros / total) * 100


# ============================================================
# STEP 9 — EVALUATION FUNCTION
# ============================================================

def evaluate_model(model):

    model.eval()

    predictions = []
    labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            predictions.extend(
                torch.argmax(
                    outputs,
                    dim=1
                ).cpu().numpy()
            )

            labels_all.extend(
                labels.numpy()
            )

    return {
        "accuracy": accuracy_score(
            labels_all,
            predictions
        ) * 100,

        "balanced_accuracy": balanced_accuracy_score(
            labels_all,
            predictions
        ) * 100,

        "macro_precision": precision_score(
            labels_all,
            predictions,
            average="macro",
            zero_division=0
        ) * 100,

        "macro_recall": recall_score(
            labels_all,
            predictions,
            average="macro",
            zero_division=0
        ) * 100,

        "macro_f1": f1_score(
            labels_all,
            predictions,
            average="macro",
            zero_division=0
        ) * 100
    }


# ============================================================
# STEP 10 — FINE-TUNING FUNCTION
# ============================================================
#
# REASONING:
# We use a small learning rate because this is fine-tuning,
# NOT training from scratch.
#
# Macro F1 is used to select the best epoch because HAM10000
# contains strong class imbalance.
# ============================================================

def fine_tune_model(
    pruned_model,
    masks,
    epochs=5,
    learning_rate=1e-5
):

    model_ft = pruned_model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model_ft.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    best_f1 = -1
    best_state = None

    for epoch in range(epochs):

        model_ft.train()

        running_loss = 0
        total_samples = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model_ft(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            # IMPORTANT:
            # Restore the pruning mask after every optimizer step.
            apply_masks(
                model_ft,
                masks
            )

            running_loss += (
                loss.item() * images.size(0)
            )

            total_samples += images.size(0)

        train_loss = (
            running_loss / total_samples
        )

        # Validation
        metrics = evaluate_model(
            model_ft
        )

        val_f1 = metrics["macro_f1"]

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Macro F1: {val_f1:.2f}%"
        )

        # Save best epoch
        if val_f1 > best_f1:

            best_f1 = val_f1

            best_state = copy.deepcopy(
                model_ft.state_dict()
            )

    # Restore best epoch
    model_ft.load_state_dict(
        best_state
    )

    # Ensure pruning mask is still applied
    apply_masks(
        model_ft,
        masks
    )

    return model_ft, best_f1


# ============================================================
# STEP 11 — RUN 10% AND 20% PRUNING + FINE-TUNING
# ============================================================
#
# We do not continue with 30% because Phase 23D already showed
# a noticeable Macro F1 degradation at that level.
# ============================================================

finetuned_models = {}
finetuning_results = []

for pruning_percent in [10, 20]:

    print("\n")
    print("=" * 70)
    print(
        f"{pruning_percent}% PRUNING + FINE-TUNING"
    )
    print("=" * 70)

    # Create fresh pruned model
    pruned_model, masks, threshold = (
        create_pruned_model(
            model,
            pruning_percent
        )
    )

    actual_sparsity = calculate_sparsity(
        pruned_model
    )

    print(
        f"Threshold: {threshold:.8f}"
    )

    print(
        f"Initial sparsity: "
        f"{actual_sparsity:.2f}%"
    )

    # Fine-tune
    fine_model, best_f1 = fine_tune_model(
        pruned_model,
        masks,
        epochs=5,
        learning_rate=1e-5
    )

    # Final evaluation
    final_metrics = evaluate_model(
        fine_model
    )

    final_sparsity = calculate_sparsity(
        fine_model
    )

    finetuned_models[
        pruning_percent
    ] = fine_model

    finetuning_results.append({
        "pruning_percent": pruning_percent,
        "threshold": threshold,
        "sparsity_percent": final_sparsity,
        **final_metrics
    })

    print("\nFinal result:")
    print(
        f"Accuracy          : "
        f"{final_metrics['accuracy']:.2f}%"
    )
    print(
        f"Balanced Accuracy : "
        f"{final_metrics['balanced_accuracy']:.2f}%"
    )
    print(
        f"Macro Precision   : "
        f"{final_metrics['macro_precision']:.2f}%"
    )
    print(
        f"Macro Recall      : "
        f"{final_metrics['macro_recall']:.2f}%"
    )
    print(
        f"Macro F1          : "
        f"{final_metrics['macro_f1']:.2f}%"
    )
    print(
        f"Sparsity          : "
        f"{final_sparsity:.2f}%"
    )


# ============================================================
# STEP 12 — FINAL SUMMARY
# ============================================================

finetuning_df = pd.DataFrame(
    finetuning_results
)

print("\n")
print("=" * 70)
print("PRUNED + FINE-TUNED MODEL RESULTS")
print("=" * 70)

display(
    finetuning_df.round(2)
)

# Save results for the research record
finetuning_df.to_csv(
    "../results/pruning_finetuning_results.csv",
    index=False
)

print(
    "\n✅ Phase 23E completed."
)

print(
    "Results saved to:"
    " ../results/pruning_finetuning_results.csv"
)

✅ Swin-Tiny architecture created.
✅ Original Swin-Tiny checkpoint loaded.

Class counts:
[ 257.  412.  891.   86.  890. 5367.  114.]

Class weights:
akiec : 4.4564
bcc   : 2.7798
bkl   : 1.2854
df    : 13.3173
mel   : 1.2868
nv    : 0.2134
vasc  : 10.0464

Prunable weight tensors: 52


10% PRUNING + FINE-TUNING
Threshold: 0.00555211
Initial sparsity: 9.38%
Epoch 1/5 | Train Loss: 0.1729 | Val Macro F1: 76.95%
Epoch 2/5 | Train Loss: 0.1305 | Val Macro F1: 76.88%
Epoch 3/5 | Train Loss: 0.1284 | Val Macro F1: 78.42%
Epoch 4/5 | Train Loss: 0.1137 | Val Macro F1: 79.14%
Epoch 5/5 | Train Loss: 0.1042 | Val Macro F1: 78.27%

Final result:
Accuracy          : 84.13%
Balanced Accuracy : 80.08%
Macro Precision   : 78.67%
Macro Recall      : 80.08%
Macro F1          : 79.14%
Sparsity          : 9.38%


20% PRUNING + FINE-TUNING
Threshold: 0.01148684
Initial sparsity: 19.16%
Epoch 1/5 | Train Loss: 0.1612 | Val Macro F1: 76.75%
Epoch 2/5 | Train Loss: 0.1526 | Val Macro F1: 76.85%
Epoch 3/5 | 

,pruning_percent,threshold,sparsity_percent,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
0,10,0.01,9.38,84.13,80.08,78.67,80.08,79.14
1,20,0.01,19.16,84.38,79.08,77.54,79.08,77.97



✅ Phase 23E completed.
Results saved to: ../results/pruning_finetuning_results.csv
